# 02. 프롬프트 기준선 — CLadder · Corr2Cause

Hume-LM과 Hume-Fast를 학습하기 전에, 범용 모델이 인과 추론 벤치마크에서 몇 점을 내는지 기준선을 잡는다. 두 방식을 함께 측정한다.

| 방식 | 설명 | 대응 모듈 |
|---|---|---|
| **Noul** | 생성 없이 답 토큰(Yes/No)의 확률만 읽는다. 정확도와 함께 보정(Brier·ECE)을 본다 | Hume-Fast |
| **CoT** | CausalCoT 6단계로 풀이를 생성하고 마지막 답을 파싱한다 | Hume-LM |

- 하드웨어: RTX PRO 6000 96GB × 2. 모델마다 GPU 한 장을 쓴다(`cuda:0`, `cuda:1`).
- CLadder는 Rung(연관·개입·반사실)별로 200문항씩, CoT는 시간이 오래 걸리므로 Rung별 50문항만 쓴다. 풀이가 길어 1024 토큰에서는 30~40%가 잘렸으므로 4096 토큰을 쓴다.
- Corr2Cause는 원본 테스트셋과 변수명 교란(refactorization) 테스트셋을 함께 본다. 교란셋 성능이 크게 떨어지면 표면 패턴을 외운 것이다.

In [ ]:
import gc

import pandas as pd
import torch

from humm.bench import datasets as D, prompts as P, runner as R
from humm.providers.hf import HFProvider

torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]

## 1. 데이터

In [ ]:
cladder = D.load_cladder(n_per_rung=200, seed=0)
cladder_cot = cladder.groupby("rung").head(50)
c2c = D.load_corr2cause("default")
c2c_ref = D.load_corr2cause("refactorization", n=1000, seed=0)

pd.DataFrame({"n": [len(cladder), len(cladder_cot), len(c2c), len(c2c_ref)],
              "pos_rate": [cladder.label.mean(), cladder_cot.label.mean(), c2c.label.mean(), c2c_ref.label.mean()]},
             index=["cladder", "cladder_cot", "corr2cause", "corr2cause_ref"]).round(3)

In [ ]:
print(cladder.prompt.iloc[0])
print("→ label:", cladder.label.iloc[0], "| rung:", cladder.rung.iloc[0], "| query:", cladder.query_type.iloc[0])

## 2. 모델

모델은 2종으로 고정한다. GPU 한 장에 하나씩 올린다.

| 모델 | GPU | 역할 |
|---|---|---|
| Qwen/Qwen3.5-4B | `cuda:0` | Hume-Fast 기반, 경량 비교군 |
| Qwen/Qwen3.5-9B | `cuda:1` | Hume-LM 주력 |

Qwen3.5는 transformers 5.2 이상이 필요하다. 기본값이 thinking 모드이지만 `HFProvider`가 `enable_thinking=False`로 호출하므로 Noul은 답 토큰을 바로 읽고, CoT는 CausalCoT 프롬프트대로 풀이를 쓴다.

In [ ]:
MODELS = {
    "Qwen/Qwen3.5-4B": "cuda:0",
    "Qwen/Qwen3.5-9B": "cuda:1",
}
BATCH = 16

In [ ]:
# Qwen3.5는 선형 어텐션 층이 있으므로, 왼쪽 패딩 배치 결과가 단건 결과와 같은지 먼저 확인한다
lm = HFProvider("Qwen/Qwen3.5-4B", device="cuda:0", batch_size=BATCH)
probe = [P.cladder_noul(r) for _, r in cladder.head(8).iterrows()]
batched = [p[0] for p in lm.option_probs(probe, R.YES_NO)]
single = [lm.option_probs([x], R.YES_NO)[0][0] for x in probe]
print(pd.DataFrame({"batched": batched, "single": single}).round(4))
del lm
gc.collect()
torch.cuda.empty_cache()

## 3. 실행

모델 하나를 올려 네 가지 평가를 모두 돌린 뒤 메모리를 비우고 다음 모델로 넘어간다. 결과는 문항 단위로 저장소 루트의 `results/baseline_qwen35.csv`에 저장한다(노트북 위치와 무관).

In [ ]:
results = []
for model_id, device in MODELS.items():
    lm = HFProvider(model_id, device=device, batch_size=BATCH)
    results += [
        R.run_noul(lm, cladder, P.cladder_noul).assign(bench="cladder"),
        R.run_noul(lm, c2c, P.corr2cause_noul).assign(bench="corr2cause"),
        R.run_noul(lm, c2c_ref, P.corr2cause_noul).assign(bench="corr2cause_ref"),
        R.run_cot(lm, cladder_cot, P.cladder_cot, max_new_tokens=4096).assign(bench="cladder"),
    ]
    del lm
    gc.collect()
    torch.cuda.empty_cache()

res = pd.concat(results, ignore_index=True)
R.save(res, "baseline_qwen35")

## 4. 결과

벤치마크별 요약과 CLadder Rung별 요약을 본다. 2장의 연구 결과대로라면 Rung 2·3(개입·반사실)에서 정확도가 떨어지고, Corr2Cause는 무작위 수준에 가까울 것이다.

In [ ]:
R.summarize(res, by=["bench"]).round(3)

In [ ]:
R.summarize(res[res.bench == "cladder"], by=["rung"]).round(3)

In [ ]:
R.summarize(res[res.bench == "cladder"], by=["query_type"]).pivot_table(index="query_type", columns=["model", "mode"], values="acc").round(3)

## 5. CoT 출력 점검

파싱 실패 문항과 오답 문항의 풀이를 직접 읽어 본다. 오류가 그래프 추출 단계에서 나는지, 추정식 도출 단계에서 나는지, 산술 단계에서 나는지가 Hume-Gym 데이터 설계의 근거가 된다.

In [ ]:
cot = res[res["mode"] == "cot"]
wrong = cot[cot.pred != cot.label]
print(len(wrong), "wrong /", len(cot))
print(wrong.iloc[0].prompt, "\n---\n", wrong.iloc[0].output[-1500:])

## 6. (선택) Anthropic API 기준선

같은 CoT 평가를 Claude로 돌려 비교한다. API는 로짓을 주지 않으므로 CoT 방식만 가능하다.

API 키는 환경변수 `ANTHROPIC_API_KEY`가 있으면 그 값을 쓰고, 없으면 입력창에서 받는다. 입력한 키는 화면과 노트북 파일에 남지 않는다.

In [ ]:
import os
from getpass import getpass

from humm.providers.anthropic_api import AnthropicProvider

os.environ["ANTHROPIC_API_KEY"] = getpass("ANTHROPIC_API_KEY: ")
api = AnthropicProvider("claude-sonnet-5-5")
api.complete(["Reply with OK."], max_new_tokens=5)

In [ ]:
res_api = R.run_cot(api, cladder_cot, P.cladder_cot, max_new_tokens=4096).assign(bench="cladder")
R.save(res_api, "baseline_api")
R.summarize(res_api, by=["rung"]).round(3)

## Mac으로 옮길 파일

실행한 노트북은 원본과 다른 이름으로 저장하고, `notebooks/`의 원본은 출력 없이 둔다.

| 파일 | 필수 여부 | 비고 |
|---|---|---|
| 이 노트북 | 필수 | `runs/<날짜>-gpu/` |
| `results/baseline_qwen35.csv` | 필수 | Qwen3.5 Noul·CoT 문항별 결과 |
| `results/baseline_api.csv` | API 셀을 돌렸을 때 | Claude CoT 결과 |
| `data/bench/` | 옮기지 않음 | 벤치마크 원본 캐시 |